# Stage 2 — Inference Acceleration
## Notebook 4: Inference Optimization

**Objective:** Master decoding parameter tuning, LLM load testing with Locust, TTFT/TPOT/throughput benchmarking, Flash Attention 2 setup, and the complete performance optimization checklist.

**Prerequisites:** Notebooks 1-3 (quantization, vLLM, SGLang), basic HTTP testing concepts.

In [ ]:
# ============================
# Cell 1: Install dependencies
# ============================
!pip install -q numpy matplotlib openai aiohttp
!pip install -q locust
!pip install -q flash-attn --no-build-isolation  # Requires CUDA 11.6+

print("Dependencies installed.")
print("Note: flash-attn requires CUDA GPU. Skip on CPU-only machines.")

## 1. Decoding Parameter Grid Search

Decoding parameters directly control the quality, diversity, and speed of LLM output. Understanding each parameter's effect is essential for production tuning.

| Parameter | Range | Effect |
|-----------|-------|--------|
| `temperature` | 0.0 - 2.0 | Lower = deterministic, Higher = creative |
| `top_p` | 0.0 - 1.0 | Nucleus sampling: cumulative probability threshold |
| `top_k` | 1 - 100 | Keep only top-K most likely tokens |
| `repetition_penalty` | 1.0 - 2.0 | >1.0 penalizes repeated tokens |
| `max_tokens` | 1 - ctx_len | Maximum generation length |

In [ ]:
# =============================================
# Cell 3: Decoding parameter grid search
# =============================================

import numpy as np
import matplotlib.pyplot as plt
from itertools import product

def grid_search_params():
    """Grid search over decoding parameters."""
    param_grid = {
        'temperature': [0.1, 0.3, 0.5, 0.7, 1.0],
        'top_p': [0.8, 0.9, 0.95, 1.0],
        'top_k': [20, 40, 50, 100],
    }
    results = []
    for temp, top_p, top_k in product(
        param_grid['temperature'],
        param_grid['top_p'],
        param_grid['top_k']
    ):
        results.append({'temperature': temp, 'top_p': top_p, 'top_k': top_k})

    print(f"Total combinations: {len(results)}")
    print(f"Sample: {results[0]}")
    print(f"\nRecommended starting point: temp=0.7, top_p=0.9, top_k=50")
    return results

grid = grid_search_params()

In [ ]:
# ==========================================
# Cell 4: Visualize parameter effects
# ==========================================

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Temperature vs diversity
temps = np.linspace(0.1, 2.0, 20)
diversity = 1 - np.exp(-temps)
axes[0].plot(temps, diversity, 'b-', linewidth=2)
axes[0].fill_between(temps, 0, diversity, alpha=0.2)
axes[0].set_xlabel('Temperature')
axes[0].set_ylabel('Output Diversity')
axes[0].set_title('Temperature vs Diversity')
axes[0].axvline(x=0.7, color='red', linestyle='--', label='Sweet spot (0.7)')
axes[0].legend()

# top_p effect
top_ps = np.linspace(0.1, 1.0, 20)
tokens_considered = 50000 * (1 - top_ps) + 10
axes[1].plot(top_ps, tokens_considered, 'g-', linewidth=2)
axes[1].set_xlabel('top_p')
axes[1].set_ylabel('Avg Tokens Considered')
axes[1].set_title('Nucleus Sampling (top_p)')
axes[1].axvline(x=0.9, color='red', linestyle='--', label='Common default (0.9)')
axes[1].legend()

# repetition_penalty
penalties = np.linspace(1.0, 2.0, 20)
repeat_prob = np.exp(-2 * (penalties - 1.0))
axes[2].plot(penalties, repeat_prob, 'r-', linewidth=2)
axes[2].set_xlabel('Repetition Penalty')
axes[2].set_ylabel('Repetition Probability')
axes[2].set_title('Repetition Penalty Effect')
axes[2].axvline(x=1.1, color='blue', linestyle='--', label='Safe default (1.1)')
axes[2].legend()

plt.tight_layout()
plt.show()

print("Parameter tuning guide:")
print("  - Creative writing: temp=0.8-1.2, top_p=0.95")
print("  - Code generation:  temp=0.1-0.3, top_p=0.9")
print("  - Chat/QA:          temp=0.5-0.7, top_p=0.9")
print("  - Factual/RAG:      temp=0.0-0.2, top_p=1.0")

## 2. Benchmarking: TTFT, TPOT, Throughput

These three metrics are the standard for evaluating LLM serving performance:

- **TTFT** (Time To First Token): Latency until the first token appears. Critical for perceived responsiveness.
- **TPOT** (Time Per Output Token): Average time between subsequent tokens (excluding first). Measures generation smoothness.
- **Throughput**: Total tokens generated per second across all concurrent requests. Measures system efficiency.

In [ ]:
# ==========================================
# Cell 6: TTFT/TPOT/Throughput Benchmark
# ==========================================

import asyncio
import aiohttp
import time
import json

class LLMBenchmark:
    """Simple LLM inference benchmark for TTFT, TPOT, throughput."""

    def __init__(self, api_base: str = "http://localhost:8000/v1"):
        self.api_base = api_base

    async def measure_single(self, prompt: str, max_tokens: int = 256):
        """Measure TTFT, TPOT, and total tokens for a single streaming request."""
        start_time = time.perf_counter()
        ttft = None
        token_count = 0

        payload = {
            "model": "default",
            "messages": [{"role": "user", "content": prompt}],
            "max_tokens": max_tokens,
            "stream": True
        }

        try:
            async with aiohttp.ClientSession() as session:
                async with session.post(
                    f"{self.api_base}/chat/completions",
                    json=payload,
                    headers={"Content-Type": "application/json"}
                ) as resp:
                    async for line in resp.content:
                        line = line.decode('utf-8').strip()
                        if line.startswith('data: '):
                            data_str = line[6:]
                            if data_str == '[DONE]':
                                break
                            try:
                                json.loads(data_str)
                                if ttft is None:
                                    ttft = time.perf_counter() - start_time
                                token_count += 1
                            except json.JSONDecodeError:
                                continue
        except Exception as e:
            return {'error': str(e), 'ttft': None, 'tpot': None, 'total_tokens': 0}

        total_time = time.perf_counter() - start_time
        tpot = (total_time - ttft) / (token_count - 1) if token_count > 1 else 0

        return {
            'ttft': ttft,
            'tpot': tpot,
            'total_tokens': token_count,
            'total_time': total_time,
            'tokens_per_second': token_count / total_time if total_time > 0 else 0
        }

    async def run_benchmark(self, prompts: list, concurrency: int = 10):
        """Run benchmark with specified concurrency."""
        sem = asyncio.Semaphore(concurrency)

        async def bounded(p):
            async with sem:
                return await self.measure_single(p)

        tasks = [bounded(p) for p in prompts]
        results = await asyncio.gather(*tasks)

        valid = [r for r in results if r.get('ttft') is not None]
        if not valid:
            return {'error': 'All requests failed'}

        ttfts = [r['ttft'] for r in valid]
        tpots = [r['tpot'] for r in valid]
        total_tokens = sum(r['total_tokens'] for r in valid)
        wall_time = max(r['total_time'] for r in valid) if valid else 1

        return {
            'num_requests': len(valid),
            'concurrency': concurrency,
            'ttft_mean': np.mean(ttfts),
            'ttft_p50': np.percentile(ttfts, 50),
            'ttft_p95': np.percentile(ttfts, 95),
            'ttft_p99': np.percentile(ttfts, 99),
            'tpot_mean': np.mean(tpots),
            'throughput_tok_per_sec': total_tokens / wall_time,
            'total_tokens': total_tokens
        }

print("LLMBenchmark class defined.")
print("\nUsage:")
print("  benchmark = LLMBenchmark('http://localhost:8000/v1')")
print("  results = await benchmark.run_benchmark(prompts, concurrency=10)")
print("  print(f\"TTFT P50: {results['ttft_p50']:.3f}s\")")
print("  print(f\"TPOT Mean: {results['tpot_mean']:.3f}s\")")
print("  print(f\"Throughput: {results['throughput_tok_per_sec']:.1f} tok/s\")")

In [ ]:
# ===========================================
# Cell 7: Visualize benchmark scaling curves
# ===========================================

# Simulated data for a 7B model on A100 across concurrency levels
concurrency_levels = [1, 2, 4, 8, 16, 32, 64, 128, 256]
throughput = [45, 88, 170, 320, 580, 950, 1400, 1800, 1950]
ttft_p50 = [0.3, 0.35, 0.4, 0.5, 0.7, 1.1, 1.8, 3.2, 6.5]
ttft_p99 = [0.5, 0.6, 0.7, 0.9, 1.4, 2.5, 4.8, 9.2, 18.0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(concurrency_levels, throughput, 'o-', color='green', linewidth=2, markersize=8)
ax1.set_xlabel('Concurrency'); ax1.set_ylabel('Throughput (tok/s)')
ax1.set_title('Throughput Scaling with Concurrency')
ax1.grid(True, alpha=0.3); ax1.set_xscale('log', base=2)
ax1.annotate('Diminishing returns\npast ~128 seqs', xy=(128, 1800), fontsize=9, color='darkgreen')

ax2.plot(concurrency_levels, ttft_p50, 's-', color='blue', linewidth=2, markersize=8, label='TTFT P50')
ax2.plot(concurrency_levels, ttft_p99, '^-', color='red', linewidth=2, markersize=8, label='TTFT P99')
ax2.set_xlabel('Concurrency'); ax2.set_ylabel('TTFT (seconds)')
ax2.set_title('Latency vs Concurrency (Tradeoff)')
ax2.legend(); ax2.grid(True, alpha=0.3); ax2.set_xscale('log', base=2)
ax2.axhline(y=2.0, color='gray', linestyle='--', alpha=0.5, label='2s SLA threshold')

plt.tight_layout(); plt.show()

print("Key observations:")
print("  1. Throughput scales near-linearly until KV cache saturates")
print("  2. P50 latency degrades gracefully; P99 spikes under high load")
print("  3. Find the knee point where throughput gain < latency cost")

## 3. Locust Load Testing for LLM APIs

**Locust** is the industry standard for HTTP load testing. Below is a complete `locustfile.py` for LLM API endpoints, simulating realistic user behavior with short queries, long-form generation, and streaming.

In [ ]:
# ==========================================
# Cell 9: Locust load testing script
# ==========================================

locustfile_content = '''
"""
Locust load test for LLM API endpoints.
Run: locust -f locustfile.py --host=http://localhost:8000
Then open http://localhost:8089 to start the test.
"""
from locust import HttpUser, task, between, events
import time
import json

class LLMUser(HttpUser):
    """Simulates users sending chat completion requests."""
    wait_time = between(0.5, 2.0)

    def on_start(self):
        self.request_count = 0
        self.total_tokens = 0

    @task(3)
    def short_query(self):
        """Quick queries -- most common."""
        start = time.time()
        with self.client.post(
            "/v1/chat/completions",
            json={
                "model": "default",
                "messages": [{"role": "user", "content": "What is 2+2?"}],
                "max_tokens": 50, "temperature": 0.7
            },
            catch_response=True
        ) as response:
            latency = (time.time() - start) * 1000
            if response.status_code == 200:
                data = response.json()
                tokens = data.get("usage", {}).get("completion_tokens", 0)
                self.total_tokens += tokens
                response.success()
            elif response.status_code == 429:
                response.failure("Rate limited")
            else:
                response.failure(f"HTTP {response.status_code}")
        self.request_count += 1

    @task(1)
    def long_query(self):
        """Long-form generation -- less common."""
        with self.client.post(
            "/v1/chat/completions",
            json={
                "model": "default",
                "messages": [{"role": "user",
                    "content": "Explain the transformer architecture in detail."}],
                "max_tokens": 512, "temperature": 0.7
            },
            catch_response=True
        ) as response:
            if response.status_code == 200:
                response.success()
            elif response.status_code != 429:
                response.failure(f"HTTP {response.status_code}")

    @task(1)
    def streaming_query(self):
        """Streaming chat -- SSE."""
        with self.client.post(
            "/v1/chat/completions",
            json={
                "model": "default",
                "messages": [{"role": "user", "content": "Tell me a joke"}],
                "max_tokens": 100, "stream": True
            },
            stream=True, catch_response=True
        ) as response:
            if response.status_code == 200:
                response.success()
            else:
                response.failure(f"HTTP {response.status_code}")


@events.quitting.add_listener
def on_quit(environment, **kwargs):
    """Print summary when Locust stops."""
    if hasattr(environment.stats, 'total'):
        total = environment.stats.total
        print(f"\\n{'='*60}")
        print(f"LOAD TEST SUMMARY")
        print(f"Total requests: {total.num_requests}")
        print(f"Failures: {total.num_failures}")
        print(f"Median response: {total.median_response_time:.0f}ms")
        print(f"95th percentile: {total.get_response_time_percentile(0.95):.0f}ms")
        print(f"99th percentile: {total.get_response_time_percentile(0.99):.0f}ms")
        print(f"RPS: {total.total_rps:.1f}")
        print(f"{'='*60}")
'''

with open('/tmp/locustfile.py', 'w') as f:
    f.write(locustfile_content)

print("locustfile.py written to /tmp/locustfile.py")
print("\nRun with: cd /tmp && locust -f locustfile.py --host=http://localhost:8000")
print("Then open http://localhost:8089 and configure:")
print("  - Number of users: 50")
print("  - Spawn rate: 10/sec")
print("  - Host: http://localhost:8000")

## 4. Context Length vs Speed Analysis

How context length affects inference speed depends on the attention implementation. Standard attention is O(n^2), while Flash Attention and PagedAttention reduce this dramatically.

In [ ]:
# =============================================
# Cell 11: Context length vs speed chart
# =============================================

context_lengths = [512, 1024, 2048, 4096, 8192, 16384, 32768]

def latency_model(ctx_len, method='standard'):
    if method == 'standard':
        return ctx_len ** 2 / 1e7 * 100  # O(n^2)
    elif method == 'flash_attn':
        return ctx_len * np.log2(ctx_len) / 1e5 * 50  # ~O(n log n)
    elif method == 'paged':
        return ctx_len / 1e3 * 20  # ~O(n)
    return ctx_len / 1e3

fig, ax = plt.subplots(figsize=(10, 6))

for method, color, ls in [
    ('standard', 'red', '-'),
    ('flash_attn', 'blue', '--'),
    ('paged', 'green', '-.')
]:
    latencies = [latency_model(cl, method) for cl in context_lengths]
    ax.plot(context_lengths, latencies, color=color, linestyle=ls,
            linewidth=2, label=method.replace('_', ' ').title())

ax.set_xlabel('Context Length (tokens)', fontsize=12)
ax.set_ylabel('Latency (ms)', fontsize=12)
ax.set_title('Attention Complexity: Context Length vs Speed', fontsize=14)
ax.legend(fontsize=11)
ax.set_xscale('log', base=2)
ax.set_yscale('log')
ax.grid(True, alpha=0.3)
ax.annotate("Standard O(n^2)\nunusable at 32k", xy=(16384, 250), fontsize=9, color='red')
ax.annotate("Flash Attn O(n log n)", xy=(8192, 45), fontsize=9, color='blue')
ax.annotate("Paged ~O(n)", xy=(32768, 50), fontsize=9, color='green')

plt.tight_layout()
plt.show()

print("Flash Attention 2 + PagedAttention make 128K+ contexts practical.")
print("Without them, inference is quadratic and quickly becomes infeasible.")

## 5. Performance Optimization Checklist

Before deploying to production, run through this optimization checklist ordered by impact-to-effort ratio.

In [ ]:
# ==========================================
# Cell 13: Optimization checklist walkthrough
# ==========================================

import pandas as pd

checklist = [
    ["1", "Enable Prefix Caching", "2-5x for shared prefixes", "Low (1 flag)", "--enable-prefix-caching"],
    ["2", "Tune max-num-batched-tokens", "+30-50% throughput", "Low", "Set to 2-4x max_model_len"],
    ["3", "Enable Flash Attention 2", "-30% VRAM, +20% speed", "Low", "attn_implementation='flash_attention_2'"],
    ["4", "Use FP8 KV Cache", "-50% KV cache VRAM", "Medium", "--kv-cache-dtype fp8"],
    ["5", "Tensor Parallelism (2 GPUs)", "~1.8x throughput", "Medium", "--tensor-parallel-size 2"],
    ["6", "Quantize to AWQ/GPTQ INT4", "-75% VRAM, +2x throughput", "Medium", "Pre-quantize with AutoAWQ"],
    ["7", "Speculative Decoding", "1.5-3x faster (single-user)", "High", "--speculative-algorithm EAGLE"],
    ["8", "Increase gpu-memory-utilization", "More concurrent requests", "Low", "Set to 0.90-0.92"],
    ["9", "Compile with torch.compile", "+10-20% speed", "Low", "Pass compiled model to engine"],
    ["10", "Use optimized CUDA kernels", "+15-30% speed", "Medium", "Marlin for GPTQ, AWQ kernels"],
]

df = pd.DataFrame(checklist, columns=["#", "Optimization", "Impact", "Effort", "How"])
print("PERFORMANCE OPTIMIZATION CHECKLIST\n" + "=" * 100)
print(df.to_string(index=False))
print("\n\nRecommended order: 1,2,3,8 (quick wins) -> 4,5,6 (medium) -> 7,9,10 (advanced)")

In [ ]:
# =======================================
# Cell 14: Environment optimization check
# =======================================

def optimization_checklist():
    """Run this checklist before deploying to production."""
    checks = []

    # Flash Attention check
    try:
        import flash_attn
        checks.append(('Flash Attention 2', True, f'v{flash_attn.__version__}'))
    except ImportError:
        checks.append(('Flash Attention 2', False, 'Not installed: pip install flash-attn --no-build-isolation'))

    # GPU check
    try:
        import torch
        gpu_count = torch.cuda.device_count()
        name = torch.cuda.get_device_name(0) if gpu_count > 0 else 'N/A'
        checks.append(('GPU Available', gpu_count > 0, f'{gpu_count} GPU(s): {name}'))
    except Exception:
        checks.append(('GPU Available', False, 'No CUDA'))

    # vLLM check
    try:
        import vllm
        checks.append(('vLLM', True, f'v{vllm.__version__}'))
    except ImportError:
        checks.append(('vLLM', False, 'Not installed'))

    # SGLang check
    try:
        import sglang
        checks.append(('SGLang', True, f'v{sglang.__version__}'))
    except ImportError:
        checks.append(('SGLang', False, 'Not installed'))

    print("=" * 70)
    print("PRODUCTION READINESS CHECKLIST")
    print("=" * 70)
    print(f"{'Item':<25} {'Status':<10} {'Detail'}")
    print("-" * 70)
    for item, status, detail in checks:
        icon = 'PASS' if status else 'TODO'
        print(f"{icon:<5} {item:<25} {'OK' if status else 'MISS':<10} {detail}")

optimization_checklist()

## 6. Continuous Batching -- Visual Explanation

Continuous batching is the single biggest inference optimization. Unlike static batching where all requests in a batch must finish before the next batch starts, continuous batching dynamically adds and removes requests.

In [ ]:
# =================================================
# Cell 16: Continuous batching visual demonstration
# =================================================

print("=" * 65)
print("Static Batching (Traditional):")
print("=" * 65)
print("""
Req A: [====A1====][====A2====]
Req B:                               [====B1====][====B2====]
Time:  |---batch 1---|---batch 2---|---batch 3---|---batch 4---|
        All requests in batch must finish before next batch starts.
        Results in idle GPU time and tail latency.
""")

print("=" * 65)
print("Continuous Batching (vLLM / SGLang):")
print("=" * 65)
print("""
Req A: [====A1====][====A2====]
Req B:          [====B1====][====B2====]
Req C:               [====C1====]            [====C2====]
Time:  |----s1----|----s2----|----s3----|----s4----|
        New requests join as soon as another finishes.
        No idle time. Higher GPU utilization.
""")

print("\nKey insight:")
print("  Continuous batching achieves 2-10x higher throughput")
print("  than static batching by eliminating idle time.")
print("  Combined with PagedAttention/RadixAttention,")
print("  it is the core reason vLLM and SGLang are so fast.")

In [ ]:
# ===========================================
# Cell 17: Simulate continuous batching speedup
# ===========================================

import random

def simulate_continuous_batching(num_requests=100, tokens_mean=150, tok_per_sec=50, max_batch=32):
    """Simulate static vs continuous batching throughput."""
    requests = [max(10, int(random.gauss(tokens_mean, 50))) for _ in range(num_requests)]

    # Static batching
    static_batches = [requests[i:i+max_batch] for i in range(0, len(requests), max_batch)]
    static_time = sum(max(batch) / tok_per_sec for batch in static_batches)

    # Continuous batching
    queue = sorted(requests, reverse=True)
    active = []
    time_elapsed = 0.0
    step = 0.1
    while queue or active:
        while len(active) < max_batch and queue:
            active.append(queue.pop(0))
        tokens_processed = tok_per_sec * step
        active = [t - tokens_processed for t in active if t > tokens_processed]
        time_elapsed += step

    speedup = static_time / time_elapsed
    print(f"Static batching:      {static_time:.1f}s")
    print(f"Continuous batching:  {time_elapsed:.1f}s")
    print(f"Speedup:              {speedup:.1f}x")

simulate_continuous_batching(100)

## 7. Flash Attention 2 Setup

Flash Attention 2 is an IO-aware exact attention algorithm that reduces memory reads/writes by tiling the computation. It provides up to 30% VRAM savings and 20% speed improvement with no accuracy loss.

In [ ]:
# ==========================================
# Cell 19: Flash Attention 2 setup guide
# ==========================================

flash_attn_guide = """
============================================================
FLASH ATTENTION 2 -- Installation & Usage
============================================================

## Installation (requires CUDA 11.6+, PyTorch 2.0+)
pip install flash-attn --no-build-isolation

## Usage with Hugging Face Transformers
from transformers import AutoModelForCausalLM
import torch

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-7B",
    attn_implementation="flash_attention_2",  # KEY FLAG
    torch_dtype=torch.float16,
    device_map="auto"
)

## vLLM & SGLang auto-detect Flash Attention -- no setup needed.

## Performance gains (7B model, A100):
# - VRAM:       14 GB -> 10 GB  (-30%)
# - Speed:      +15-25% faster
# - Context:    Supports ~2x longer sequences
# - Accuracy:   Mathematically identical to standard attention

## How it works:
# 1. Tiles Q, K, V into blocks that fit in SRAM
# 2. Computes softmax incrementally (online softmax)
# 3. Avoids writing full attention matrix to HBM
# 4. Result: IO-aware, exact, faster
"""

print(flash_attn_guide)

# Check if Flash Attention is available
try:
    import flash_attn
    print(f"\nFlash Attention 2 is INSTALLED (v{flash_attn.__version__})")
except ImportError:
    print("\nFlash Attention 2 is NOT installed.")
    print("Install with: pip install flash-attn --no-build-isolation")

## 8. Exercise: Complete Benchmarking Script

Build a complete benchmarking script that measures TTFT, TPOT, and throughput across multiple concurrency levels, then plots the scaling curve and saves a JSON report.

In [ ]:
# ==================================================
# Cell 21: Exercise -- Complete the benchmarking script
# ==================================================

async def full_benchmark(api_url: str, prompts: list):
    """
    TODO: Complete this full benchmarking script.

    It should:
    1. Create an LLMBenchmark instance
    2. Run benchmarks at concurrency levels [1, 5, 10, 20, 50]
    3. Collect and store results for each level
    4. Plot a scaling curve (concurrency vs throughput + latency)
    5. Save results to benchmark_report.json
    """
    benchmark = LLMBenchmark(api_url)
    results_by_concurrency = {}

    for concurrency in [1, 5, 10, 20, 50]:
        # TODO: Run benchmark at this concurrency level
        # result = await benchmark.run_benchmark(prompts, concurrency)
        # results_by_concurrency[concurrency] = result
        # print(f"Concurrency {concurrency}: {result}")
        pass  # REPLACE THIS

    # TODO: Plot scaling curve (throughput left y-axis, TTFT right y-axis)
    # TODO: Save results to benchmark_report.json

    return results_by_concurrency


sample_prompts = [
    "What is machine learning?",
    "Explain quantum computing in simple terms.",
    "Write a haiku about programming.",
    "What are the three laws of thermodynamics?",
    "Describe the TCP/IP model.",
] * 10  # 50 prompts total

print("Exercise TODO:")
print("  1. Complete the full_benchmark function")
print("  2. Run against your LLM API endpoint")
print("  3. Generate the scaling curve plot")
print("  4. Save benchmark_report.json")
print("\nHint: Use LLMBenchmark class from Cell 6.")
print("Hint: For plotting, use matplotlib with twin axes.")
print("Hint: json.dump() with indent=2 for readable output.")

In [ ]:
# ==========================================
# Cell 22: Exercise solution (reference)
# ==========================================

print("=" * 60)
print("REFERENCE SOLUTION -- Complete Benchmarking Script")
print("=" * 60)
print("""
async def full_benchmark(api_url, prompts):
    benchmark = LLMBenchmark(api_url)
    results = {}

    for concurrency in [1, 5, 10, 20, 50]:
        print(f"Testing concurrency={concurrency}...")
        result = await benchmark.run_benchmark(prompts, concurrency)
        results[concurrency] = result
        if 'error' not in result:
            print(f"  TTFT P50: {result['ttft_p50']:.3f}s, "
                  f"Throughput: {result['throughput_tok_per_sec']:.1f} tok/s")

    # Plot scaling curve
    concurrencies = list(results.keys())
    throughputs = [results[c].get('throughput_tok_per_sec', 0) for c in concurrencies]
    ttfts = [results[c].get('ttft_p50', 0) for c in concurrencies]

    fig, ax1 = plt.subplots(figsize=(10, 5))
    ax1.set_xlabel('Concurrency')
    ax1.set_ylabel('Throughput (tok/s)', color='tab:green')
    ax1.plot(concurrencies, throughputs, 'o-', color='tab:green', linewidth=2)
    ax1.tick_params(axis='y', labelcolor='tab:green')

    ax2 = ax1.twinx()
    ax2.set_ylabel('TTFT P50 (s)', color='tab:red')
    ax2.plot(concurrencies, ttfts, 's-', color='tab:red', linewidth=2)
    ax2.tick_params(axis='y', labelcolor='tab:red')

    plt.title('Scaling Curve: Throughput vs Latency Tradeoff')
    fig.tight_layout()
    plt.savefig('scaling_curve.png', dpi=100)
    plt.show()

    # Save report
    with open('benchmark_report.json', 'w') as f:
        json.dump({str(k): v for k, v in results.items()}, f, indent=2, default=str)
    print("Report saved to benchmark_report.json")

    return results
""")
print("\nCompare your solution with the reference above.")

## 9. Key Takeaways

1. **Decoding parameter tuning matters**: temp=0.7, top_p=0.9, top_k=50 is a good starting point. Lower temperature for factual tasks, higher for creative ones.

2. **Always benchmark TTFT, TPOT, and Throughput** -- they measure different aspects of performance and you cannot optimize all three simultaneously.

3. **Locust is the standard** for LLM API load testing. Simulate realistic traffic patterns (quick queries frequent, long generations rare).

4. **Context length has a massive impact** on latency. Flash Attention 2 (O(n log n)) and PagedAttention (~O(n)) make long contexts practical.

5. **Start with quick wins** from the optimization checklist: enable prefix caching, tune batch parameters, and enable Flash Attention 2 -- these give the biggest gains for the least effort.

6. **Continuous batching** is the single biggest optimization, achieving 2-10x throughput over static batching by eliminating idle time.

7. **Flash Attention 2** saves 30% VRAM and adds 20% speed with mathematically identical results -- it should be enabled by default.

### Stage 2 Complete

You now understand the full LLM inference acceleration stack: quantization (compressing models), vLLM (high-throughput serving), SGLang (structured generation), and inference optimization (tuning, benchmarking, and checklist).

### What's Next

Apply these techniques to your Stage 3 production deployment and measure the improvement against your baseline.